# Topic 6 - Your First LLM API Call

This notebook makes an LLM request visible at the HTTP layer. We will build a request, inspect a response, read provider documentation like engineers, and triage common HTTP errors.

**Rule:** use no SDK in this notebook. The live example uses Python's standard library only.

## Learning objectives

By the end, you should be able to:

- identify the HTTP method, endpoint, headers, authentication, and JSON body
- represent conversation history with a `messages` list and roles
- inspect response metadata, content, usage, and stop reason
- find exact syntax in official provider documentation
- triage `400`, `401`, and `429` before changing code randomly

The dry-run cells work without an API key. The live call is explicitly opt-in.

## 1. The request as a data structure

A provider-specific API call has several layers:

```text
POST + endpoint
        + headers
          + authentication
            + JSON body
              -> provider
              <- HTTP status + JSON response
```

The endpoint, authentication header, model name, and response schema are provider-specific. Always verify them in current official documentation.

In [1]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

messages = [
    {
        'role': 'user',
        'content': 'Explain statelessness in one sentence.',
    }
]

request_body = {
    'model': 'provider-model-name',
    'max_tokens': 128,
    'messages': messages,
}

assert isinstance(request_body['messages'], list)
assert request_body['messages'][0]['role'] == 'user'
print(json.dumps(request_body, indent=2))

{
  "model": "provider-model-name",
  "max_tokens": 128,
  "messages": [
    {
      "role": "user",
      "content": "Explain statelessness in one sentence."
    }
  ]
}


### Practice: explain each field

Before running a request, answer these questions in your own words:

1. Why is this usually a `POST` request?
2. Which field identifies the model?
3. Why is `messages` a list rather than one string?
4. Which part belongs to authentication, and which part belongs to model input?
5. Which values must be checked against provider documentation?

## 2. Messages and roles

A conversation is structured data supplied by your program. The API does not need to remember previous turns if you resend the history. A message's `role` describes what kind of participant it represents; `content` contains the message data.

A multi-turn history might look like this:

```python
messages = [
    {'role': 'user', 'content': 'What is Java?'},
    {'role': 'assistant', 'content': 'Java is a programming language.'},
    {'role': 'user', 'content': 'What about Spring?'},
]
```

Do not assume that every provider supports exactly the same roles or content shape. Confirm the endpoint schema.

In [2]:
conversation = [
    {'role': 'user', 'content': 'What is Java?'},
    {'role': 'assistant', 'content': 'Java is a programming language.'},
    {'role': 'user', 'content': 'What about Spring?'},
]

assert [message['role'] for message in conversation] == ['user', 'assistant', 'user']
assert conversation[-1]['content'] == 'What about Spring?'
print(json.dumps(conversation, indent=2))

[
  {
    "role": "user",
    "content": "What is Java?"
  },
  {
    "role": "assistant",
    "content": "Java is a programming language."
  },
  {
    "role": "user",
    "content": "What about Spring?"
  }
]


## 3. Provider syntax: two concrete request shapes

The concepts are shared, but exact syntax differs. These examples are documentation-reading exercises; do not run them until you verify the current model and required fields in the linked provider docs.

### OpenAI-style request

- endpoint: `https://api.openai.com/v1/responses`
- authentication: `Authorization: Bearer <key>`
- content type: `application/json`
- request fields and response shape: verify in the official Responses API documentation

### Anthropic-style request

- endpoint: `https://api.anthropic.com/v1/messages`
- authentication: `x-api-key: <key>`
- content type: `application/json`
- API version header: verify the current required `anthropic-version` value
- request fields and response shape: verify in the official Messages API documentation

In [3]:
def build_anthropic_request(api_key, model, messages):
    url = 'https://api.anthropic.com/v1/messages'
    headers = {
        'x-api-key': api_key,
        'anthropic-version': '2023-06-01',
        'content-type': 'application/json',
    }
    body = {
        'model': model,
        'max_tokens': 128,
        'messages': messages,
    }
    return Request(
        url,
        data=json.dumps(body).encode('utf-8'),
        headers=headers,
        method='POST',
    )

dry_run_request = build_anthropic_request('not-a-real-key', 'verify-this-model-in-docs', messages)
assert dry_run_request.method == 'POST'
assert dry_run_request.get_header('Content-type') == 'application/json'
assert dry_run_request.full_url.endswith('/v1/messages')
print(dry_run_request.method, dry_run_request.full_url)
print({key: value for key, value in dry_run_request.header_items() if key.lower() != 'x-api-key'})

POST https://api.anthropic.com/v1/messages
{'Anthropic-version': '2023-06-01', 'Content-type': 'application/json'}


Notice that the key is a header, not part of the prompt or JSON body. The code above intentionally uses a fake key and never prints a real credential.

## 4. Send a raw HTTP request with the standard library

The function below uses `urllib`, so there is no provider SDK. It returns both the HTTP status and parsed JSON body. HTTP failures are captured instead of being hidden.

Set `RUN_LIVE_API_CALL = True` only after setting the matching environment variable and confirming the current provider documentation.

In [4]:
def post_json(request, timeout=30):
    try:
        with urlopen(request, timeout=timeout) as response:
            raw_body = response.read().decode('utf-8')
            return response.status, json.loads(raw_body)
    except HTTPError as error:
        raw_body = error.read().decode('utf-8', errors='replace')
        try:
            error_body = json.loads(raw_body)
        except json.JSONDecodeError:
            error_body = {'raw_body': raw_body}
        return error.code, error_body
    except URLError as error:
        return None, {'network_error': str(error.reason)}

RUN_LIVE_API_CALL = False

if RUN_LIVE_API_CALL:
    api_key = os.environ['ANTHROPIC_API_KEY']
    live_request = build_anthropic_request(
        api_key=api_key,
        model=os.environ['ANTHROPIC_MODEL'],
        messages=[{'role': 'user', 'content': 'Explain statelessness in one sentence.'}],
    )
    status_code, response_body = post_json(live_request)
    print('HTTP status:', status_code)
    print(json.dumps(response_body, indent=2))
else:
    print('Dry run: set RUN_LIVE_API_CALL = True and ANTHROPIC_API_KEY/ANTHROPIC_MODEL to make a real call.')

Dry run: set RUN_LIVE_API_CALL = True and ANTHROPIC_API_KEY/ANTHROPIC_MODEL to make a real call.


## 5. Read every field of the response

Do not reduce a response immediately to one generated sentence. First inspect its schema. Common fields include an identifier, model information, generated content, a stop reason, and usage metadata. Exact names and nesting vary by provider.

For Anthropic Messages responses, text is commonly nested under `content` blocks, while token counts are under `usage`.

In [5]:
sample_response = {
    'id': 'msg_example',
    'type': 'message',
    'role': 'assistant',
    'model': 'provider-model-name',
    'content': [{'type': 'text', 'text': 'An API is stateless when each request includes the context it needs.'}],
    'stop_reason': 'end_turn',
    'usage': {'input_tokens': 18, 'output_tokens': 15},
}

print('id:', sample_response.get('id'))
print('model:', sample_response.get('model'))
print('content:', sample_response.get('content'))
print('stop_reason:', sample_response.get('stop_reason'))
print('usage:', sample_response.get('usage'))

text_blocks = [
    block['text']
    for block in sample_response.get('content', [])
    if block.get('type') == 'text'
]
assert text_blocks
assert sample_response['usage']['input_tokens'] > 0
assert sample_response['stop_reason'] == 'end_turn'

id: msg_example
model: provider-model-name
content: [{'type': 'text', 'text': 'An API is stateless when each request includes the context it needs.'}]
stop_reason: end_turn
usage: {'input_tokens': 18, 'output_tokens': 15}


### HTTP status is not the stop reason

`200` means the HTTP request was processed successfully. `stop_reason` describes why generation ended. For example, a successful response might have `200` and `stop_reason = 'max_tokens'`. That is different from an HTTP error such as `400`, `401`, or `429`.

Usage matters because input and output tokens affect cost, limits, and context consumption.

## 6. Read provider documentation as an engineer

For any new provider, answer these questions from official documentation:

1. What is the exact endpoint and HTTP method?
2. Which authentication header is required?
3. Which content type and version headers are required?
4. What are the minimum required JSON fields?
5. What is the exact `messages` and role schema?
6. Where are generated content, usage, and stop information in the response?
7. What does the error response contain?
8. Are there request IDs, retry headers, or rate-limit headers?

Sources:
- [OpenAI API docs](https://developers.openai.com/api/docs)
- [Anthropic API docs](https://platform.claude.com/docs)

## 7. First-level error triage

| Status | First question | Investigation direction |
| --- | --- | --- |
| `400` | What is wrong with my request? | JSON, required fields, types, model, endpoint |
| `401` | What is wrong with authentication? | Key, header, provider, account, expiry |
| `429` | What limit have I hit? | Rate, quota, spend, retry timing, `Retry-After` |

This is first-level triage, not the final diagnosis. Always read the provider's complete error body.

In [6]:
def classify_http_status(status_code):
    if status_code == 400:
        return 'request problem'
    if status_code == 401:
        return 'authentication problem'
    if status_code == 429:
        return 'rate or usage limit problem'
    if status_code is not None and 200 <= status_code < 300:
        return 'HTTP success; inspect the response and stop reason'
    return 'read the provider documentation and error body'

assert classify_http_status(400) == 'request problem'
assert classify_http_status(401) == 'authentication problem'
assert classify_http_status(429) == 'rate or usage limit problem'

for status_code in [200, 400, 401, 429, 500]:
    print(status_code, '->', classify_http_status(status_code))

200 -> HTTP success; inspect the response and stop reason
400 -> request problem
401 -> authentication problem
429 -> rate or usage limit problem
500 -> read the provider documentation and error body


In [7]:
sample_errors = {
    400: {'error': {'type': 'invalid_request_error', 'message': 'messages is required'}},
    401: {'error': {'type': 'authentication_error', 'message': 'invalid API key'}},
    429: {'error': {'type': 'rate_limit_error', 'message': 'try again later'}},
}

for status_code, body in sample_errors.items():
    message = body.get('error', {}).get('message', 'No message supplied')
    print(status_code, classify_http_status(status_code), '|', message)

assert 'messages' in sample_errors[400]['error']['message']
assert 'key' in sample_errors[401]['error']['message']
assert 'again' in sample_errors[429]['error']['message']

400 request problem | messages is required
401 authentication problem | invalid API key
429 rate or usage limit problem | try again later


### Error exercises

For each status, state the first thing you would inspect and one change you would avoid making blindly:

- `401` with `invalid API key`
- `400` with `messages is required`
- `429` with a `Retry-After` header

Expected direction: authentication for `401`, request construction for `400`, and rate/quota/retry policy for `429`.

## 8. Completion check

You are done with this topic when you can explain all of the following without an SDK:

- how the endpoint, headers, authentication, and JSON body form one HTTP request
- why conversation history is a list of role/content messages
- where to find content, usage, metadata, and stop reason in a response
- why `200` and `end_turn` describe different layers
- why `400`, `401`, and `429` send you to different parts of the investigation

Final mental model:

```text
Python program
  -> HTTP POST: endpoint + headers + JSON
  <- HTTP response: status + JSON + metadata
  -> inspect fields, then classify failures
```